In [22]:
import yfinance as yf

In [23]:
df = yf.download("Reliance.NS",start="2022-03-01",end="2026-03-01")
df.columns = df.columns.droplevel(1)

[*********************100%***********************]  1 of 1 completed


In [24]:
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2022-03-02,1091.369141,1092.483982,1059.814064,1062.202855,21471786
2022-03-03,1082.155151,1098.785820,1078.401303,1092.028893,10186748
2022-03-04,1058.153198,1075.784852,1055.787154,1070.643199,10805667
2022-03-07,1019.022278,1052.033400,1011.036774,1036.835909,17983310
2022-03-08,1017.179382,1021.957085,991.926235,1006.259101,21289374


In [25]:
df.tail()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-02-23,1428.000000,1434.900024,1418.300049,1425.000000,7758856
2026-02-24,1428.800049,1433.300049,1415.000000,1425.300049,12529409
2026-02-25,1398.500000,1440.500000,1393.500000,1435.000000,10728792
2026-02-26,1406.800049,1412.900024,1391.900024,1398.500000,16683858
2026-02-27,1393.900024,1410.400024,1388.099976,1398.000000,12031440


In [26]:
return_=(df['Close'].shift(-1)-df['Close'])/df['Close']
df['Target']=(return_>0).astype(int)

In [27]:
print(df['Target'].value_counts())
print(df['Target'].head(10))
print(return_.head(10))

Target
1    505
0    484
Name: count, dtype: int64
Date
2022-03-02    0
2022-03-03    0
2022-03-04    0
2022-03-07    0
2022-03-08    1
2022-03-09    1
2022-03-10    1
2022-03-11    1
2022-03-14    0
2022-03-15    1
Name: Target, dtype: int64
Date
2022-03-02   -0.008443
2022-03-03   -0.022180
2022-03-04   -0.036980
2022-03-07   -0.001808
2022-03-08    0.053053
2022-03-09    0.016206
2022-03-10    0.002884
2022-03-11    0.008211
2022-03-14   -0.022903
2022-03-15    0.016882
Name: Close, dtype: float64


In [28]:
df.dropna(inplace=True)
df.shape

(989, 6)

In [29]:
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

In [30]:
df[['Close','SMA_20','SMA_50']].head(60)

Price,Close,SMA_20,SMA_50
Date,,,
2022-03-02,1091.369141,NaN,NaN
2022-03-03,1082.155151,NaN,NaN
2022-03-04,1058.153198,NaN,NaN
2022-03-07,1019.022278,NaN,NaN
2022-03-08,1017.179382,NaN,NaN
2022-03-09,1071.143799,NaN,NaN
2022-03-10,1088.502441,NaN,NaN
2022-03-11,1091.642090,NaN,NaN
2022-03-14,1100.605835,NaN,NaN


In [31]:
daily_change=df['Close'].diff()
gain=daily_change.clip(lower=0)
loss=daily_change.clip(upper=0).abs()
avg_gain=gain.rolling(14).mean()
avg_loss=loss.rolling(14).mean()
RS=avg_gain/avg_loss
df['RSI']=100-100/(1+RS)

In [32]:
df.head(60)

Price,Close,High,Low,Open,Volume,Target,SMA_20,SMA_50,RSI
Date,,,,,,,,,
2022-03-02,1091.369141,1092.483982,1059.814064,1062.202855,21471786,0,NaN,NaN,NaN
2022-03-03,1082.155151,1098.785820,1078.401303,1092.028893,10186748,0,NaN,NaN,NaN
2022-03-04,1058.153198,1075.784852,1055.787154,1070.643199,10805667,0,NaN,NaN,NaN
2022-03-07,1019.022278,1052.033400,1011.036774,1036.835909,17983310,0,NaN,NaN,NaN
2022-03-08,1017.179382,1021.957085,991.926235,1006.259101,21289374,1,NaN,NaN,NaN
2022-03-09,1071.143799,1077.013475,1016.565078,1017.406956,20414228,1,NaN,NaN,NaN
2022-03-10,1088.502441,1094.303876,1073.828372,1092.028820,17980493,1,NaN,NaN,NaN
2022-03-11,1091.642090,1097.034080,1081.131313,1081.131313,12257423,1,NaN,NaN,NaN
2022-03-14,1100.605835,1103.131228,1080.198570,1086.068246,8340678,0,NaN,NaN,NaN


In [33]:
print(df['RSI'].describe())
print(df['RSI'].head(20))

count    975.000000
mean      51.485630
std       17.083185
min        7.722604
25%       38.741114
50%       52.553386
75%       64.495286
max       97.563393
Name: RSI, dtype: float64
Date
2022-03-02          NaN
2022-03-03          NaN
2022-03-04          NaN
2022-03-07          NaN
2022-03-08          NaN
2022-03-09          NaN
2022-03-10          NaN
2022-03-11          NaN
2022-03-14          NaN
2022-03-15          NaN
2022-03-16          NaN
2022-03-17          NaN
2022-03-21          NaN
2022-03-22          NaN
2022-03-23    61.601918
2022-03-24    66.019016
2022-03-25    72.914568
2022-03-28    86.082285
2022-03-29    86.760406
2022-03-30    84.790013
Name: RSI, dtype: float64


In [34]:
ema_12 = df['Close'].ewm(span=12).mean()
ema_26 = df['Close'].ewm(span=26).mean()

df['MACD'] = ema_12 - ema_26
df['Signal_Line'] = df['MACD'].ewm(span=9).mean()
df['MACD_Histogram'] = df['MACD']-df['Signal_Line']

In [35]:
df[['MACD', 'Signal_Line', 'MACD_Histogram']].head(30)

Price,MACD,Signal_Line,MACD_Histogram
Date,,,
2022-03-02,0.000000,0.000000,0.000000
2022-03-03,-0.206724,-0.114847,-0.091877
2022-03-04,-1.007707,-0.480773,-0.526934
2022-03-07,-2.760120,-1.252909,-1.507210
2022-03-08,-3.727581,-1.989068,-1.738512
2022-03-09,-1.931540,-1.973475,0.041935
2022-03-10,0.039925,-1.463937,1.503862
2022-03-11,1.510532,-0.749116,2.259648
2022-03-14,2.950640,0.105546,2.845094


In [36]:
print(df.columns)
print(df.head(2))

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Target', 'SMA_20', 'SMA_50',
       'RSI', 'MACD', 'Signal_Line', 'MACD_Histogram'],
      dtype='str', name='Price')
Price             Close         High          Low         Open    Volume  \
Date                                                                       
2022-03-02  1091.369141  1092.483982  1059.814064  1062.202855  21471786   
2022-03-03  1082.155151  1098.785820  1078.401303  1092.028893  10186748   

Price       Target  SMA_20  SMA_50  RSI      MACD  Signal_Line  MACD_Histogram  
Date                                                                            
2022-03-02       0     NaN     NaN  NaN  0.000000     0.000000        0.000000  
2022-03-03       0     NaN     NaN  NaN -0.206724    -0.114847       -0.091877  


In [37]:
std_20=df['Close'].rolling(20).std()
df['bb_Upper']=df['SMA_20']+2*std_20
df['bb_Lower']=df['SMA_20']-2*std_20
df['bb_Width']=(df['bb_Upper']-df['bb_Lower'])/df['SMA_20']

In [41]:
df['Momentum']=df['Close']-df['Close'].shift(10)
df['Volatility']=return_.rolling(20).std

In [44]:
df.dropna(inplace=True)
print(df.shape)

(940, 17)
